# 01 — Data Preparation

**Project:** Urban Heat Island Satellite Analysis — London, UK  
**Author:** Zari Syed  
**Date:** 2025

---

## Overview

This notebook handles all data loading and preprocessing steps prior to analysis. It covers:

1. Importing required libraries and setting up project paths
2. Loading the Greater London boundary shapefile
3. Inspecting raw Landsat 8 (2015) and Landsat 9 (2023) band files
4. Reprojecting vector boundaries to match raster CRS
5. Clipping all relevant bands to the Greater London extent
6. Saving processed outputs to `data/processed/`

**Prerequisites:**  
- Raw Landsat scenes placed in `data/raw/landsat_2015/` and `data/raw/landsat_2023/`  
- GLA boundary shapefile in `shapefiles/`  
- All dependencies installed (`pip install -r requirements.txt`)

## 1. Imports and Configuration

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import geopandas as gpd
from shapely.geometry import mapping

warnings.filterwarnings('ignore')

print(f"rasterio version: {rasterio.__version__}")
print(f"geopandas version: {gpd.__version__}")

In [ ]:
# ── Project root (one level up from this notebook) ──────────────────────────
PROJECT_ROOT = Path('..').resolve()

# ── Input directories ────────────────────────────────────────────────────────
RAW_2015     = PROJECT_ROOT / 'data' / 'raw' / 'landsat_2015'
RAW_2023     = PROJECT_ROOT / 'data' / 'raw' / 'landsat_2023'
SHAPEFILES   = PROJECT_ROOT / 'shapefiles'

# ── Output directories ───────────────────────────────────────────────────────
PROCESSED    = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── Target CRS: British National Grid ────────────────────────────────────────
TARGET_CRS = CRS.from_epsg(27700)

print("Directories configured.")
print(f"  Raw 2015 : {RAW_2015}")
print(f"  Raw 2023 : {RAW_2023}")
print(f"  Processed: {PROCESSED}")

## 2. Load Greater London Boundary

The GLA boundary shapefile defines the clip extent for all raster operations. We reproject it to British National Grid (EPSG:27700) which is the standard for UK spatial analysis and ensures accurate area measurements.

In [ ]:
# Load Greater London boundary (dissolve boroughs to single polygon)
london_boundary_path = SHAPEFILES / 'greater_london_boundary.shp'
london_gdf = gpd.read_file(london_boundary_path)

print(f"Loaded boundary CRS: {london_gdf.crs}")
print(f"Number of features: {len(london_gdf)}")
london_gdf.head()

In [ ]:
# Reproject to British National Grid
london_bng = london_gdf.to_crs(TARGET_CRS)

# Dissolve to a single geometry (outer boundary only)
london_dissolved = london_bng.dissolve()

print(f"Reprojected CRS: {london_dissolved.crs}")
print(f"Total area: {london_dissolved.area.values[0] / 1e6:.1f} km²")

# Extract geometry for masking
london_geom = [mapping(london_dissolved.geometry.values[0])]

## 3. Inspect Raw Landsat Files

USGS Collection 2 Level-2 products use a standardised naming convention. We identify the relevant band files programmatically:

- **SR_B4** — Red (0.64–0.67 µm) — used for NDVI
- **SR_B5** — Near Infrared (0.85–0.88 µm) — used for NDVI
- **ST_B10** — Thermal Infrared (10.60–11.19 µm) — used for LST
- **QA_PIXEL** — Cloud/quality flags

In [ ]:
def find_band_file(scene_dir: Path, band_suffix: str) -> Path:
    """
    Locate a Landsat band file by its suffix within a scene directory.
    USGS naming: LC08_L2SP_201024_20150803_*_*_{band_suffix}.TIF
    """
    matches = list(scene_dir.glob(f'*_{band_suffix}.TIF'))
    if not matches:
        raise FileNotFoundError(
            f"No file matching '*_{band_suffix}.TIF' found in {scene_dir}"
        )
    if len(matches) > 1:
        print(f"  Warning: multiple matches for {band_suffix}, using first: {matches[0].name}")
    return matches[0]


# Discover band files for both years
bands_2015 = {
    'red'     : find_band_file(RAW_2015, 'SR_B4'),
    'nir'     : find_band_file(RAW_2015, 'SR_B5'),
    'thermal' : find_band_file(RAW_2015, 'ST_B10'),
    'qa'      : find_band_file(RAW_2015, 'QA_PIXEL'),
}

bands_2023 = {
    'red'     : find_band_file(RAW_2023, 'SR_B4'),
    'nir'     : find_band_file(RAW_2023, 'SR_B5'),
    'thermal' : find_band_file(RAW_2023, 'ST_B10'),
    'qa'      : find_band_file(RAW_2023, 'QA_PIXEL'),
}

print("2015 band files:")
for k, v in bands_2015.items():
    print(f"  {k:10s}: {v.name}")

print("\n2023 band files:")
for k, v in bands_2023.items():
    print(f"  {k:10s}: {v.name}")

In [ ]:
# Inspect metadata of a representative band
with rasterio.open(bands_2015['thermal']) as src:
    print("=== 2015 ST_B10 metadata ===")
    print(f"  CRS         : {src.crs}")
    print(f"  Resolution  : {src.res} m")
    print(f"  Dimensions  : {src.width} × {src.height} pixels")
    print(f"  Bounds      : {src.bounds}")
    print(f"  Data type   : {src.dtypes[0]}")
    print(f"  NoData value: {src.nodata}")

## 4. Cloud Masking

The QA_PIXEL band encodes cloud and cloud-shadow flags using bit positions. We mask pixels flagged as cloud (bit 3) or cloud shadow (bit 4) following USGS documentation.

In [ ]:
def build_cloud_mask(qa_path: Path) -> np.ndarray:
    """
    Create a boolean mask from QA_PIXEL band.
    True = valid pixel, False = cloud or cloud shadow.

    Bits masked:
        Bit 3 (8)   — Cloud
        Bit 4 (16)  — Cloud shadow
        Bit 1 (2)   — Dilated cloud
    """
    with rasterio.open(qa_path) as src:
        qa = src.read(1).astype(np.uint16)

    cloud_bits = (1 << 1) | (1 << 3) | (1 << 4)  # bits 1, 3, 4
    valid_mask = (qa & cloud_bits) == 0

    cloud_pct = (~valid_mask).sum() / valid_mask.size * 100
    print(f"  Cloud-affected pixels: {cloud_pct:.1f}%")
    return valid_mask


print("2015 cloud mask:")
cloud_mask_2015 = build_cloud_mask(bands_2015['qa'])

print("\n2023 cloud mask:")
cloud_mask_2023 = build_cloud_mask(bands_2023['qa'])

## 5. Reproject Rasters to British National Grid

Landsat scenes are delivered in UTM projection. We reproject to EPSG:27700 (BNG) to align with the GLA boundary data and ensure consistency throughout the analysis.

In [ ]:
def reproject_raster(src_path: Path, dst_path: Path, target_crs: CRS) -> None:
    """
    Reproject a raster to target_crs and save to dst_path.
    Uses bilinear resampling.
    """
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            'crs'      : target_crs,
            'transform': transform,
            'width'    : width,
            'height'   : height,
            'compress' : 'lzw',
        })

        with rasterio.open(dst_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source      =rasterio.band(src, i),
                    destination =rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs     =src.crs,
                    dst_transform=transform,
                    dst_crs     =target_crs,
                    resampling  =Resampling.bilinear,
                )
    print(f"  Saved: {dst_path.name}")


print("Reprojecting 2015 bands...")
for band_name, src_path in bands_2015.items():
    dst = PROCESSED / f'2015_{band_name}_bng.tif'
    reproject_raster(src_path, dst, TARGET_CRS)

print("\nReprojecting 2023 bands...")
for band_name, src_path in bands_2023.items():
    dst = PROCESSED / f'2023_{band_name}_bng.tif'
    reproject_raster(src_path, dst, TARGET_CRS)

## 6. Clip to Greater London Boundary

We clip each reprojected band to the Greater London boundary polygon, masking pixels outside the study area with NoData. This reduces file sizes and focuses all downstream computations on the study area.

In [ ]:
def clip_raster_to_boundary(
    src_path : Path,
    dst_path : Path,
    geometry : list,
    nodata   : float = -9999.0
) -> None:
    """
    Clip a raster to a geometry and write to dst_path.
    geometry: list of GeoJSON-like dicts (as returned by shapely's mapping())
    """
    with rasterio.open(src_path) as src:
        clipped, transform = rio_mask(
            src,
            geometry,
            crop      =True,
            nodata    =nodata,
            filled    =True,
        )
        meta = src.meta.copy()
        meta.update({
            'driver'   : 'GTiff',
            'height'   : clipped.shape[1],
            'width'    : clipped.shape[2],
            'transform': transform,
            'nodata'   : nodata,
            'compress' : 'lzw',
        })

        with rasterio.open(dst_path, 'w', **meta) as dst:
            dst.write(clipped)

    print(f"  Clipped: {dst_path.name}  (shape: {clipped.shape[1]}×{clipped.shape[2]})")


# Clip all reprojected bands
bng_bands = {
    '2015_red'     : PROCESSED / '2015_red_bng.tif',
    '2015_nir'     : PROCESSED / '2015_nir_bng.tif',
    '2015_thermal' : PROCESSED / '2015_thermal_bng.tif',
    '2023_red'     : PROCESSED / '2023_red_bng.tif',
    '2023_nir'     : PROCESSED / '2023_nir_bng.tif',
    '2023_thermal' : PROCESSED / '2023_thermal_bng.tif',
}

print("Clipping to Greater London boundary...")
for name, src_path in bng_bands.items():
    dst_path = PROCESSED / f'{name}_london.tif'
    clip_raster_to_boundary(src_path, dst_path, london_geom)

print("\nAll bands clipped successfully.")

## 7. Verify Outputs

Confirm all processed files exist and share consistent spatial properties.

In [ ]:
processed_files = sorted(PROCESSED.glob('*_london.tif'))

print(f"Processed files in {PROCESSED}:")
print(f"{'File':<45} {'CRS':<12} {'Dimensions':<20} {'Resolution'}")
print("-" * 90)

for f in processed_files:
    with rasterio.open(f) as src:
        dims = f"{src.width}×{src.height}"
        res  = f"{src.res[0]:.1f}m"
        crs  = str(src.crs).split(':')[-1]
        print(f"{f.name:<45} EPSG:{crs:<8} {dims:<20} {res}")

print("\n✓ Data preparation complete. Proceed to 02_analysis.ipynb")

---

## Summary

At the end of this notebook, `data/processed/` contains the following London-clipped, BNG-projected raster files:

| File | Description |
|---|---|
| `2015_red_bng_london.tif` | Landsat 8 Band 4 (Red), 2015 |
| `2015_nir_bng_london.tif` | Landsat 8 Band 5 (NIR), 2015 |
| `2015_thermal_bng_london.tif` | Landsat 8 Band 10 (ST), 2015 |
| `2023_red_bng_london.tif` | Landsat 9 Band 4 (Red), 2023 |
| `2023_nir_bng_london.tif` | Landsat 9 Band 5 (NIR), 2023 |
| `2023_thermal_bng_london.tif` | Landsat 9 Band 10 (ST), 2023 |

These files are inputs to `02_analysis.ipynb`.